<a href="https://colab.research.google.com/github/mayayonathan/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item within an order; for example, an order with two items (order 1, Cheeseburger and Soda) becomes two rows, each of which is still tagged with its vendor_id and order.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)

assert len(joined) == len(lines)
print(joined['_merge'].value_counts())

unmatched = joined[joined['_merge'] == 'left_only']
print()
print('unmatched vendor_id:', unmatched['vendor_id'].unique().tolist())
print('units attached to it:', unmatched['qty'].sum())
joined

_merge
both          3
left_only     1
right_only    0
Name: count, dtype: int64

unmatched vendor_id: ['V-07']
units attached to it: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# Decision: V-07 has no entry in the vendor list, but it still represents
# real units sold, so it is labeled rather than dropped -- dropping it would
# silently remove 3 of the 7 units from the report.
joined['name'] = joined['name'].fillna('Unknown (' + joined['vendor_id'] + ')')

by_vendor = joined.groupby('name')['qty'].sum().sort_values(ascending=False)
by_vendor

,qty
name,
Hoos Burgers,3
Unknown (V-07),3
Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
by_zone = joined.groupby('zone')['qty'].sum().sort_values(ascending=False)
print(by_zone)

print()
print('zone total:', by_zone.sum(), 'vs grand total:', joined['qty'].sum())
print("The difference (3 units) is V-07's line, since the unmatched vendor")
print("has no zone to group into -- groupby drops rows with a NaN key by default.")

zone
A    4
Name: qty, dtype: int64

zone total: 4 vs grand total: 7
The difference (3 units) is V-07's line, since the unmatched vendor
has no zone to group into -- groupby drops rows with a NaN key by default.


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# json_normalize crashes on a missing record_path key, so build rows by hand
# with .get(), which returns None instead of raising.
rows = []
for o in ragged_json:
    for line in o.get('lines', []):
        rows.append({
            'order': o['order'],
            'vendor_id': o['vendor_id'],
            'item': line.get('item'),
            'qty': line.get('qty'),   # stays None/NaN -- never becomes 0
        })

ragged = pd.DataFrame(rows)
print(ragged)
print()
print(ragged.dtypes)
print('missing qty:', ragged['qty'].isna().sum())
print("Order 5 contributes no rows at all, since it has no line items to flatten.")
print("Order 6's Fries row keeps qty as NaN -- distinguishable from a real 0,")
print("which fillna(0) would have erased.")

   order vendor_id   item  qty
0      4      V-01   Soda  2.0
1      6      V-01  Fries  NaN

order          int64
vendor_id     object
item          object
qty          float64
dtype: object
missing qty: 1
Order 5 contributes no rows at all, since it has no line items to flatten.
Order 6's Fries row keeps qty as NaN -- distinguishable from a real 0,
which fillna(0) would have erased.


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** "How many Hot Dogs did we sell" would require manually scanning each order's contents because each row would require a distinct column for each potential item (or a list crammed into one cell). Any query about items, numbers, or income by item becomes a group-and-sum rather than a parsing difficulty when there is only one row per line item.

**b)** We don't know how many units were on that line if the amount is absent; this could be due to a data entry error or the field failing to log. A quantity of 0 indicates that the answer is zero and we do know. Every "unknown" line becomes "confirmed zero" if you confuse them with `fillna(0)`, which understates actual totals (such as Q4's zone total losing V-07's units) without leaving any indication that something was estimated rather than measured. Later on, no one reading the report can distinguish between the two cases, so they are unable to determine whether to trust the zero or look into the gap.